# Lab 25: The Grammar of AI

## Vectors, embeddings, attention, and trainable matrix machines

This lab is a computational capstone for the book. We will not build a huge AI model. Instead, we will build small transparent versions of the core linear-algebra objects used in modern AI.

You will explore:

1. embeddings and cosine search;
2. document-term matrices and semantic geometry;
3. neural network layers as matrix machines;
4. softmax and classification scores;
5. attention as matrix multiplication;
6. low-rank structure and hidden factors;
7. high-dimensional geometry.

The goal is to see that AI is not magic. It is built from vectors, matrices, dot products, normalization, projections, optimization, and geometry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(7)

## 1. Embeddings: objects become vectors

An embedding is a vector representation of an object. In real AI systems, embeddings are learned from large datasets. Here we create a toy 2D embedding space by hand so that we can see the geometry.

In [ ]:
words = np.array(["matrix", "vector", "eigenvalue", "dog", "cat", "pizza", "pasta", "neural", "attention", "gradient"])
E2 = np.array([
    [0.9, 0.8],
    [0.8, 0.9],
    [0.7, 0.75],
    [-0.8, 0.5],
    [-0.75, 0.55],
    [0.2, -0.9],
    [0.25, -0.85],
    [0.85, 0.25],
    [0.9, 0.15],
    [0.75, 0.35]
])

plt.figure(figsize=(7,6))
plt.scatter(E2[:,0], E2[:,1], s=80)
for w, (x,y) in zip(words, E2):
    plt.text(x+0.02, y+0.02, w, fontsize=11)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Toy 2D embedding space")
plt.xlabel("coordinate 1")
plt.ylabel("coordinate 2")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()

### Student task

Which words appear close together? Which groups form clusters? What does this say about the geometry of meaning?

## 2. Cosine similarity and vector search

A search engine based on embeddings ranks objects by similarity to a query vector. Cosine similarity compares direction:

$$
\operatorname{cosine}(u,v)=\frac{u\cdot v}{\|u\|\|v\|}.
$$

In [ ]:
def normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-12)

def cosine_scores(X, q):
    return normalize_rows(X) @ (q / np.linalg.norm(q))

query = np.array([0.9, 0.7])
scores = cosine_scores(E2, query)
ranking = pd.DataFrame({"word": words, "cosine_similarity": scores}).sort_values("cosine_similarity", ascending=False)
ranking

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(E2[:,0], E2[:,1], s=80)
plt.arrow(0, 0, query[0], query[1], head_width=0.04, length_includes_head=True)
plt.text(query[0]+0.03, query[1]+0.03, "query", fontsize=12)
for w, (x,y) in zip(words, E2):
    plt.text(x+0.02, y+0.02, w, fontsize=11)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Vector search: rank by angle to the query")
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.show()

## 3. From documents to matrices

Now we create a tiny document-term matrix. Each row is a document. Each column is a vocabulary word.

In [ ]:
documents = [
    "linear algebra studies vectors matrices and eigenvectors",
    "machine learning uses matrices vectors gradients and data",
    "cats and dogs are common pets",
    "pizza pasta and bread are foods",
    "attention and neural networks use vectors and matrices",
    "PCA eigenvectors and SVD reveal hidden structure"
]

vocab = sorted(set(" ".join(doc.lower() for doc in documents).split()))
X = np.zeros((len(documents), len(vocab)), dtype=float)
for i, doc in enumerate(documents):
    for token in doc.lower().split():
        X[i, vocab.index(token)] += 1

dtm = pd.DataFrame(X, columns=vocab, index=[f"doc {i+1}" for i in range(len(documents))])
dtm

In [ ]:
# Search with a query document
query_text = "vectors matrices neural learning"
q = np.zeros(len(vocab))
for token in query_text.lower().split():
    if token in vocab:
        q[vocab.index(token)] += 1

scores = cosine_scores(X, q)
result = pd.DataFrame({"document": documents, "score": scores}).sort_values("score", ascending=False)
result

### Reflection

This simple bag-of-words method does not understand grammar or context, but it already shows the basic linear algebra of text search: documents are rows of a matrix, and a query is a vector.

## 4. Neural-network layers as matrix machines

A neural layer computes

$$
h = \sigma(Wx+b).
$$

Here $W$ is a matrix, $b$ is a vector, and $\sigma$ is a nonlinear activation such as ReLU.

In [ ]:
def relu(z):
    return np.maximum(0, z)

x = np.array([2.0, -1.0, 0.5])
W = np.array([
    [1.0, -1.0, 0.5],
    [-0.5, 2.0, 1.0],
    [0.25, 0.25, -1.0],
    [1.0, 1.0, 1.0]
])
b = np.array([0.1, -0.2, 0.3, 0.0])

z = W @ x + b
h = relu(z)
print("pre-activation z =", z)
print("activation h =", h)

In [ ]:
# Visualize each row of W as a detector for input x
row_scores = W @ x
plt.figure(figsize=(7,4))
plt.bar(np.arange(len(row_scores)), row_scores)
plt.axhline(0, linewidth=1)
plt.xticks(np.arange(len(row_scores)), [f"row {i+1}" for i in range(len(row_scores))])
plt.ylabel("dot product with input")
plt.title("Rows of W act as pattern detectors")
plt.show()

## 5. Softmax turns scores into probabilities

A classifier often produces one score per class. Softmax converts these scores into positive numbers that sum to 1.

In [ ]:
def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

class_names = np.array(["math", "animals", "food", "AI"])
logits = np.array([2.1, -0.5, 0.2, 1.7])
probs = softmax(logits)
pd.DataFrame({"class": class_names, "score": logits, "softmax_probability": probs})

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(class_names, probs)
plt.ylim(0,1)
plt.title("Softmax probabilities")
plt.ylabel("probability")
plt.show()

## 6. Attention from scratch

Attention begins with a matrix of token embeddings $X$. It creates queries, keys, and values:

$$
Q=XW_Q, \qquad K=XW_K, \qquad V=XW_V.
$$

Then it forms dot-product scores:

$$
S=\frac{QK^T}{\sqrt{d_k}}.
$$

After row-wise softmax,

$$
A=\operatorname{softmax}(S),
$$

the output is

$$
Z=AV.
$$

In [ ]:
def softmax_rows(S):
    S = S - S.max(axis=1, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=1, keepdims=True)

# Four token embeddings in R^3
Xtok = np.array([
    [1.0, 0.0, 0.2],   # token 1
    [0.8, 0.1, 0.1],   # token 2
    [0.0, 1.0, 0.3],   # token 3
    [0.1, 0.8, 0.4]    # token 4
])

WQ = rng.normal(size=(3,2))
WK = rng.normal(size=(3,2))
WV = rng.normal(size=(3,3))

Q = Xtok @ WQ
K = Xtok @ WK
V = Xtok @ WV
S = Q @ K.T / np.sqrt(Q.shape[1])
A = softmax_rows(S)
Z = A @ V

print("Attention weights A:")
print(A)
print("\nOutput Z:")
print(Z)

In [ ]:
plt.figure(figsize=(5,4))
plt.imshow(A)
plt.colorbar(label="attention weight")
plt.xticks(range(4), ["tok1", "tok2", "tok3", "tok4"])
plt.yticks(range(4), ["tok1", "tok2", "tok3", "tok4"])
plt.title("Attention matrix")
plt.show()

### Student task

Look at each row of the attention matrix. Which tokens does each token attend to most strongly? Why does each row sum to 1?

## 7. Low-rank structure: hidden factors

Many AI systems rely on hidden low-dimensional structure. We create a low-rank matrix and recover its dominant singular values.

In [ ]:
n_users, n_items, rank = 30, 20, 3
U_true = rng.normal(size=(n_users, rank))
V_true = rng.normal(size=(n_items, rank))
R = U_true @ V_true.T + 0.2 * rng.normal(size=(n_users, n_items))

U, s, Vt = np.linalg.svd(R, full_matrices=False)

plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(s)+1), s, marker='o')
plt.xlabel("index")
plt.ylabel("singular value")
plt.title("Singular values reveal hidden low-rank structure")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def rank_k_approx(U, s, Vt, k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

errors = []
for k in range(1, 11):
    Rk = rank_k_approx(U, s, Vt, k)
    errors.append(np.linalg.norm(R - Rk, 'fro') / np.linalg.norm(R, 'fro'))

plt.figure(figsize=(6,4))
plt.plot(range(1,11), errors, marker='o')
plt.xlabel("rank k")
plt.ylabel("relative reconstruction error")
plt.title("Low-rank approximation error")
plt.grid(True, alpha=0.3)
plt.show()

## 8. High-dimensional geometry

Embeddings often live in high dimensions. Random high-dimensional vectors tend to be almost orthogonal. This fact affects search, clustering, and interpretation.

In [ ]:
dims = [2, 5, 10, 50, 100, 500]
mean_abs_cos = []

for d in dims:
    A_rand = rng.normal(size=(2000, d))
    B_rand = rng.normal(size=(2000, d))
    A_rand = A_rand / np.linalg.norm(A_rand, axis=1, keepdims=True)
    B_rand = B_rand / np.linalg.norm(B_rand, axis=1, keepdims=True)
    cos = np.sum(A_rand * B_rand, axis=1)
    mean_abs_cos.append(np.mean(np.abs(cos)))

plt.figure(figsize=(7,4))
plt.plot(dims, mean_abs_cos, marker='o')
plt.xscale('log')
plt.xlabel("dimension")
plt.ylabel("mean absolute cosine")
plt.title("Random vectors become nearly orthogonal in high dimensions")
plt.grid(True, alpha=0.3)
plt.show()

pd.DataFrame({"dimension": dims, "mean_abs_cosine": mean_abs_cos})

## 9. Mini-project: build a tiny AI-style retrieval system

Complete the following tasks.

1. Add at least five more documents to the document list.
2. Build a document-term matrix.
3. Add a query.
4. Rank documents using cosine similarity.
5. Try a second query and compare the ranking.
6. Explain where the method works and where it fails.

Optional extension: implement TF-IDF weights instead of raw counts.

In [ ]:
# Starter code for the mini-project
my_documents = [
    "linear algebra powers machine learning",
    "dogs cats and pets are animals",
    "gradient descent trains neural networks",
    "matrix factorization helps recommendation systems",
    "pizza pasta and cooking recipes"
]

# TODO: add more documents
# TODO: build vocabulary
# TODO: build document-term matrix
# TODO: create a query vector
# TODO: rank by cosine similarity


## 10. Final reflection

Write a short paragraph answering:

> Which linear algebra idea from the book appears most often in this lab? Why?

Possible answers include dot product, matrix multiplication, projection, SVD, optimization, or high-dimensional geometry.